# Download and parse ID minter failure files

Downloads all `.ids.failures.ndjson` files from S3 and extracts the failed `row_id` values.

In [5]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [ ]:
import boto3
import json
from pathlib import Path
from datetime import datetime, timezone

S3_BUCKET = "wellcomecollection-platform-id-minter"
S3_PREFIX = "prod/id_minter"
SUFFIX = ".ids.failures.ndjson"
CUT_OFF = datetime(2026, 3, 19, tzinfo=timezone.utc) # that's when we switched over

DATA_DIR = Path.cwd().resolve() / "data" / "id_minter_failures"
DATA_DIR.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3")

# List all objects matching the suffix, modified after CUT_OFF date
keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(SUFFIX) and obj["LastModified"] > CUT_OFF:
            keys.append(obj["Key"])

print(f"Found {len(keys)} failure files (modified after {CUT_OFF.isoformat()})")
for k in keys[:10]:
    print(f"  {k}")
if len(keys) > 10:
    print(f"  ... and {len(keys) - 10} more")

In [ ]:
# Download all failure files
for key in keys:
    local_path = DATA_DIR / Path(key).name
    if not local_path.exists():
        print(f"Downloading {key}")
        s3.download_file(S3_BUCKET, key, str(local_path))
    else:
        print(f"Already downloaded {local_path.name}")

print(f"\nDownloaded {len(keys)} files to {DATA_DIR}")

In [ ]:
# Parse all failure files and extract row_ids
failed_ids = []

for path in sorted(DATA_DIR.glob(f"*{SUFFIX}")):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            failed_ids.append(record["row_id"])

print(f"Total failed IDs: {len(failed_ids)}")
print(f"Unique failed IDs: {len(set(failed_ids))}")
print(f"\nFirst 20 failed IDs:")
for row_id in failed_ids[:20]:
    print(f"  {row_id}")

In [ ]:
# Re-invoke the id-minter lambda with failed IDs in batches of 10
from datetime import datetime, timezone

LAMBDA_ARN = "arn:aws:lambda:eu-west-1:760097843905:function:catalogue-2025-10-02-id-minter"
BATCH_SIZE = 10

lambda_client = boto3.client("lambda", region_name="eu-west-1")
job_id = f"rerun_{datetime.now(timezone.utc).isoformat()}"
unique_failed_ids = list(set(failed_ids))

batches = [
    unique_failed_ids[i : i + BATCH_SIZE]
    for i in range(0, len(unique_failed_ids), BATCH_SIZE)
]

print(f"Job ID: {job_id}")
print(f"Invoking {len(batches)} batches of up to {BATCH_SIZE} IDs ({len(unique_failed_ids)} unique IDs total)\n")

results = []
for i, batch in enumerate(batches):
    payload = {
        "sourceIdentifiers": batch,
        "jobId": job_id,
    }
    response = lambda_client.invoke(
        FunctionName=LAMBDA_ARN,
        InvocationType="RequestResponse",
        Payload=json.dumps(payload),
    )
    result = json.loads(response["Payload"].read())
    results.append(result)
    status = response.get("StatusCode", "?")
    n_success = len(result.get("successes", []))
    n_fail = len(result.get("failures", []))
    print(f"Batch {i + 1}/{len(batches)}: status={status}, successes={n_success}, failures={n_fail}")

total_successes = sum(len(r.get("successes", [])) for r in results)
total_failures = sum(len(r.get("failures", [])) for r in results)
print(f"\nDone. Total successes: {total_successes}, total failures: {total_failures}")